In [1]:
import requests
import time
import pandas as pd

# =========================
# CONFIG
# =========================
BASE_URL = "https://caching.graphql.imdb.com/"
OPERATION_NAME = "TitleReviewsRefine"
PAGE_SIZE = 25

HEADERS = {
    "accept": "application/graphql+json, application/json",
    "accept-language": "en-US,en;q=0.9",
    "content-type": "application/json",
    "origin": "https://www.imdb.com",
    "priority": "u=1, i",
    "user-agent": "Mozilla/5.0"
}

QUERY = """
query TitleReviewsRefine($const: ID!, $filter: ReviewsFilter, $first: Int!, $sort: ReviewsSort, $after: ID) {
  title(id: $const) {
    reviews(filter: $filter, first: $first, sort: $sort, after: $after) {
      edges {
        node {
          id
          author {
            nickName
          }
          authorRating
          helpfulness {
            upVotes
            downVotes
          }
          submissionDate
          text {
            originalText {
              plainText
            }
          }
          summary {
            originalText
          }
        }
      }
      pageInfo {
        hasNextPage
        endCursor
      }
    }
  }
}
"""

In [2]:
# =========================
# MCU IDS
# =========================
mcu_ids = [

    # ===== Phase One =====
    "tt0371746",  # Iron Man (2008)
    "tt0800080",  # The Incredible Hulk (2008)
    "tt1228705",  # Iron Man 2 (2010)
    "tt0800369",  # Thor (2011)
    "tt0458339",  # Captain America: The First Avenger (2011)
    "tt0848228",  # The Avengers (2012)

    # ===== Phase Two =====
    "tt1300854",  # Iron Man 3 (2013)
    "tt1981115",  # Thor: The Dark World (2013)
    "tt1843866",  # Captain America: The Winter Soldier (2014)
    "tt2015381",  # Guardians of the Galaxy (2014)
    "tt2395427",  # Avengers: Age of Ultron (2015)
    "tt0478970",  # Ant-Man (2015)

    # ===== Phase Three =====
    "tt3498820",  # Captain America: Civil War (2016)
    "tt1211837",  # Doctor Strange (2016)
    "tt3896198",  # Guardians of the Galaxy Vol. 2 (2017)
    "tt2250912",  # Spider-Man: Homecoming (2017)
    "tt3501632",  # Thor: Ragnarok (2017)
    "tt1825683",  # Black Panther (2018)
    "tt4154756",  # Avengers: Infinity War (2018)
    "tt5095030",  # Ant-Man and the Wasp (2018)
    "tt4154796",  # Avengers: Endgame (2019)
    "tt6320628",  # Spider-Man: Far From Home (2019)
    "tt4154664",  # Captain Marvel (2019)

    # ===== Phase Four =====
    "tt3480822",  # Black Widow (2021)
    "tt9376612",  # Shang-Chi and the Legend of the Ten Rings (2021)
    "tt9032400",  # Eternals (2021)
    "tt10872600", # Spider-Man: No Way Home (2021)
    "tt9419884",  # Doctor Strange in the Multiverse of Madness (2022)
    "tt10648342", # Thor: Love and Thunder (2022)
    "tt9114286",  # Black Panther: Wakanda Forever (2022)

    # ===== Phase Five =====
    "tt10954600", # Ant-Man and the Wasp: Quantumania (2023)
    "tt6791350",  # Guardians of the Galaxy Vol. 3 (2023)
    "tt10676048", # The Marvels (2023)
    "tt6263850",  # Deadpool & Wolverine (2024)
    "tt14513804", # Captain America: Brave New World (2025)
    "tt20969586", # Thunderbolts* (2025)

    # ===== Phase Six =====
    "tt10676052", # The Fantastic Four: First Steps (2025)
]

# Optional: map IDs to movie names
movie_map = {
    "tt0371746": "Iron Man",
    "tt0800080": "The Incredible Hulk",
    "tt1228705": "Iron Man 2",
    "tt0800369": "Thor",
    "tt0458339": "Captain America: The First Avenger",
    "tt0848228": "The Avengers",
    "tt1300854": "Iron Man 3",
    "tt1981115": "Thor: The Dark World",
    "tt1843866": "Captain America: The Winter Soldier",
    "tt2015381": "Guardians of the Galaxy",
    "tt2395427": "Avengers: Age of Ultron",
    "tt0478970": "Ant-Man",
    "tt3498820": "Captain America: Civil War",
    "tt1211837": "Doctor Strange",
    "tt3896198": "Guardians of the Galaxy Vol. 2",
    "tt2250912": "Spider-Man: Homecoming",
    "tt3501632": "Thor: Ragnarok",
    "tt1825683": "Black Panther",
    "tt4154756": "Avengers: Infinity War",
    "tt5095030": "Ant-Man and the Wasp",
    "tt4154796": "Avengers: Endgame",
    "tt6320628": "Spider-Man: Far From Home",
    "tt4154664": "Captain Marvel",
    "tt3480822": "Black Widow",
    "tt9376612": "Shang-Chi and the Legend of the Ten Rings",
    "tt9032400": "Eternals",
    "tt10872600": "Spider-Man: No Way Home",
    "tt9419884": "Doctor Strange in the Multiverse of Madness",
    "tt10648342": "Thor: Love and Thunder",
    "tt9114286": "Black Panther: Wakanda Forever",
    "tt10954600": "Ant-Man and the Wasp: Quantumania",
    "tt6791350": "Guardians of the Galaxy Vol. 3",
    "tt10676048": "The Marvels",
    "tt6263850": "Deadpool & Wolverine",
    "tt14513804": "Captain America: Brave New World",
    "tt20969586": "Thunderbolts*",
    "tt10676052": "The Fantastic Four: First Steps",
}

In [3]:
# =========================
# HELPERS
# =========================
def get_variables(movie_id, after_cursor=None, sort_by="HELPFULNESS_SCORE", sort_order="DESC"):
    variables = {
        "const": movie_id,
        "filter": {},
        "first": PAGE_SIZE,
        "sort": {
            "by": sort_by,
            "order": sort_order
        }
    }
    if after_cursor:
        variables["after"] = after_cursor
    return variables


def fetch_page(movie_id, after_cursor=None, sort_by="HELPFULNESS_SCORE", sort_order="DESC", timeout=30):
    payload = {
        "query": QUERY,
        "operationName": OPERATION_NAME,
        "variables": get_variables(movie_id, after_cursor, sort_by, sort_order)
    }

    response = requests.post(
        BASE_URL,
        headers=HEADERS,
        json=payload,
        timeout=timeout
    )
    response.raise_for_status()
    return response.json()


def extract_reviews(data, movie_id):
    movie_title = movie_map.get(movie_id, None)

    edges = (
        data.get("data", {})
        .get("title", {})
        .get("reviews", {})
        .get("edges", [])
    )

    rows = []
    for edge in edges:
        node = edge.get("node", {}) or {}

        rows.append({
            "movie_id": movie_id,
            "movie_title": movie_title,
            "review_id": node.get("id"),
            "author": (node.get("author") or {}).get("nickName"),
            "author_rating": node.get("authorRating"),
            "upvotes": (node.get("helpfulness") or {}).get("upVotes"),
            "downvotes": (node.get("helpfulness") or {}).get("downVotes"),
            "submission_date": node.get("submissionDate"),
            "summary": (node.get("summary") or {}).get("originalText"),
            "content": ((node.get("text") or {}).get("originalText") or {}).get("plainText"),
        })

    return rows


def get_all_reviews_for_movie(movie_id, delay=0.75, max_pages=None, sort_by="HELPFULNESS_SCORE", sort_order="DESC"):
    all_reviews = []
    after_cursor = None
    page = 1

    while True:
        try:
            data = fetch_page(
                movie_id=movie_id,
                after_cursor=after_cursor,
                sort_by=sort_by,
                sort_order=sort_order
            )
        except Exception as e:
            print(f"[ERROR] {movie_id} page {page}: {e}")
            break

        if "errors" in data:
            print(f"[ERROR] IMDb returned errors for {movie_id} page {page}: {data['errors']}")
            break

        page_reviews = extract_reviews(data, movie_id)
        all_reviews.extend(page_reviews)

        reviews_obj = (
            data.get("data", {})
            .get("title", {})
            .get("reviews", {})
        )

        page_info = reviews_obj.get("pageInfo", {}) or {}
        has_next = page_info.get("hasNextPage", False)
        end_cursor = page_info.get("endCursor")

        print(f"{movie_map.get(movie_id, movie_id)} | page {page} | reviews collected so far: {len(all_reviews)}")

        if not has_next:
            break

        if max_pages is not None and page >= max_pages:
            break

        if not end_cursor:
            break

        after_cursor = end_cursor
        page += 1
        time.sleep(delay)

    return all_reviews


def clean_reviews_df(df):
    if df.empty:
        return df

    if "content" in df.columns:
        df["content"] = (
            df["content"]
            .fillna("")
            .astype(str)
            .str.replace("\n", " ", regex=False)
            .str.replace("\r", " ", regex=False)
            .str.strip()
        )

    if "summary" in df.columns:
        df["summary"] = (
            df["summary"]
            .fillna("")
            .astype(str)
            .str.replace("\n", " ", regex=False)
            .str.replace("\r", " ", regex=False)
            .str.strip()
        )

    df = df.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
    df = df[df["content"].str.len() > 0].reset_index(drop=True)

    return df

In [ ]:
# =========================
# MAIN CRAWL
# =========================
all_reviews = []
movie_stats = []

for movie_id in mcu_ids:
    movie_title = movie_map.get(movie_id, movie_id)
    print(f"\n=== Fetching reviews for {movie_title} ({movie_id}) ===")

    movie_reviews = get_all_reviews_for_movie(
        movie_id=movie_id,
        delay=0.75,          # increase if you get blocked
        max_pages=None,      # set to e.g. 5 for testing
        sort_by="HELPFULNESS_SCORE",
        sort_order="DESC"
    )

    all_reviews.extend(movie_reviews)

    movie_stats.append({
        "movie_id": movie_id,
        "movie_title": movie_title,
        "n_reviews": len(movie_reviews)
    })

    print(f"Finished {movie_title}: {len(movie_reviews)} reviews")

In [6]:
# =========================
# TO DATAFRAME
# =========================
df_reviews = pd.DataFrame(all_reviews)
df_reviews = clean_reviews_df(df_reviews)

df_stats = pd.DataFrame(movie_stats).sort_values(
    by=["n_reviews", "movie_title"],
    ascending=[False, True]
).reset_index(drop=True)

print("\nTotal reviews:", len(df_reviews))


Total reviews: 90310


In [7]:
# =========================
# SAVE
# =========================
df_reviews.to_csv("mcu_imdb_reviews.csv", index=False)
df_reviews.to_parquet("mcu_imdb_reviews.parquet", index=False)

df_stats.to_csv("mcu_review_counts.csv", index=False)

print("\nSaved:")
print("- mcu_imdb_reviews.csv")
print("- mcu_imdb_reviews.parquet")
print("- mcu_review_counts.csv")


Saved:
- mcu_imdb_reviews.csv
- mcu_imdb_reviews.parquet
- mcu_review_counts.csv


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

review_path = 'data/mcu_imdb_reviews.parquet'


### Basic Cleaning

In [39]:
# df_reviews is your full crawled dataframe
df = pd.read_parquet(review_path).dropna()

df = df.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
df = df[df["content"].notna() & (df["content"].str.strip() != "")].copy()
df["author_rating"] = pd.to_numeric(df["author_rating"], errors="coerce")

# keep only ratings 1-10
df = df[df["author_rating"].isin(range(1, 11))].copy()
df["author_rating"] = df["author_rating"].astype(int)

In [40]:
TARGET_N = 1000
RATINGS = list(range(1, 11))
TARGET_PER_RATING = TARGET_N // len(RATINGS)   # 100 each

rng_seed = 42
sampled_parts = []

for rating in RATINGS:
    df_r = df[df["author_rating"] == rating].copy()

    movies = sorted(df_r["movie_title"].dropna().unique())
    n_movies = len(movies)

    if n_movies == 0:
        continue

    target_r = min(TARGET_PER_RATING, len(df_r))

    base_per_movie = target_r // n_movies
    remainder = target_r % n_movies

    movie_targets = {m: base_per_movie for m in movies}
    for m in movies[:remainder]:
        movie_targets[m] += 1

    rating_samples = []

    # first pass: take each movie's target
    for movie in movies:
        df_rm = df_r[df_r["movie_title"] == movie]
        n_take = min(movie_targets[movie], len(df_rm))

        if n_take > 0:
            s = df_rm.sample(n=n_take, random_state=rng_seed)
            rating_samples.append(s)

    if rating_samples:
        df_rating_sample = pd.concat(rating_samples, ignore_index=False)
    else:
        df_rating_sample = pd.DataFrame(columns=df.columns)

    # second pass: top up within this rating if some movies were too small
    shortfall = target_r - len(df_rating_sample)

    if shortfall > 0:
        remaining = df_r.loc[~df_r.index.isin(df_rating_sample.index)]
        if len(remaining) > 0:
            topup = remaining.sample(
                n=min(shortfall, len(remaining)),
                random_state=rng_seed
            )
            df_rating_sample = pd.concat([df_rating_sample, topup], ignore_index=False)

    sampled_parts.append(df_rating_sample)

df_label = pd.concat(sampled_parts, ignore_index=False)
df_label = df_label.drop_duplicates(subset=["review_id"]).reset_index(drop=True)

In [41]:
current_n = len(df_label)

if current_n < TARGET_N:
    remaining_global = df.loc[~df["review_id"].isin(df_label["review_id"])]
    if len(remaining_global) > 0:
        extra = remaining_global.sample(
            n=min(TARGET_N - current_n, len(remaining_global)),
            random_state=rng_seed
        )
        df_label = pd.concat([df_label, extra], ignore_index=True)

elif current_n > TARGET_N:
    # trim inside each rating to keep rating balance
    final_parts = []
    for rating in RATINGS:
        df_r = df_label[df_label["author_rating"] == rating]
        n_take = min(TARGET_PER_RATING, len(df_r))
        final_parts.append(df_r.sample(n=n_take, random_state=rng_seed))
    df_label = pd.concat(final_parts, ignore_index=True)


In [45]:
df_label = df_label[[
    "review_id",
    "movie_id",
    "movie_title",
    "author_rating",
    "submission_date",
    "summary",
    "content"
]].copy()

df_label["subjective_label"] = ""
df_label["polarity_label"] = ""
df_label["notes"] = ""

print("Final size:", len(df_label))
print("\nRating counts:")
print(df_label["author_rating"].value_counts().sort_index())

print("\nMovie counts:")
print(df_label["movie_title"].value_counts().sort_values())

df_label.to_csv("data/mcu_manual_label_1000_rating_priority.csv", index=False)

Final size: 1000

Rating counts:
author_rating
1     100
2     100
3     100
4     100
5     100
6     100
7     100
8     100
9     100
10    100
Name: count, dtype: int64

Movie counts:
movie_title
Thunderbolts*                                  20
Spider-Man: Homecoming                         20
The Avengers                                   20
The Fantastic Four: First Steps                20
The Incredible Hulk                            20
Spider-Man: No Way Home                        20
Thor                                           20
Thor: Love and Thunder                         20
Thor: Ragnarok                                 20
The Marvels                                    20
Thor: The Dark World                           20
Avengers: Infinity War                         30
Captain America: The First Avenger             30
Captain America: Civil War                     30
Captain America: Brave New World               30
Black Widow                                    30


In [43]:
df_label['author_rating'].value_counts()

author_rating
1     100
2     100
3     100
4     100
5     100
6     100
7     100
8     100
9     100
10    100
Name: count, dtype: int64

In [17]:
import pandas as pd

In [26]:
df

,movie_id,movie_title,review_id,author,author_rating,upvotes,downvotes,submission_date,summary,content
0,tt0371746,Iron Man,rw4383391,bjoernidler,8.0,284,16,2018-10-07,Still one of the best MCU-movies,In my attempt to rewatch all the MCU-movies ch...
1,tt0371746,Iron Man,rw3067309,TheLittleSongbird,9.0,105,5,2014-08-12,A Marvel superhero film done with class,When it comes to ranking the Marvel superhero(...
2,tt0371746,Iron Man,rw3765080,jhudson-11704,9.0,113,9,2017-07-27,"Don't waste your life, Stark","With a B-list superhero, a risky lead actor, a..."
3,tt0371746,Iron Man,rw6571911,repojack,8.0,57,3,2021-02-07,This is where it all began. Queue AC/DC.,MCU 2021 Marathon MCU #1: Iron Man was my fa...
4,tt0371746,Iron Man,rw2918335,SnoopyStyle,8.0,35,3,2013-12-07,RDJ is Iron Man,Tony Stark (Robert Downey Jr.) is a hard playi...
...,...,...,...,...,...,...,...,...,...,...
90305,tt10676052,The Fantastic Four: First Steps,rw11285078,agnesliversedge,2.0,0,0,2026-03-04,feels like passive aggressiv pro life.,I love the consept of a superhero that gets pr...
90306,tt10676052,The Fantastic Four: First Steps,rw11242873,yatopa,NaN,0,0,2026-02-21,The Fantastic Four: First Steps marks a signif...,"The story centers on Reed Richards, Sue Storm,..."
90307,tt10676052,The Fantastic Four: First Steps,rw11256087,vapoy-5,NaN,0,0,2026-02-23,"Directed by Matt Shakman, the film focuses les...","The story centers on Reed Richards, Sue Storm,..."
90308,tt10676052,The Fantastic Four: First Steps,rw11240907,jepotol,NaN,0,0,2026-02-21,The Fantastic Four: First Steps marks an impor...,"The story centers on Reed Richards, Sue Storm,..."


In [18]:
df_label = pd.read_csv('data/mcu_manual_label_1000_rating_priority.csv')
df = pd.read_csv('data/mcu_imdb_reviews.csv')

In [19]:
df_remaining = df[~df["review_id"].isin(df_label["review_id"])].copy()

In [22]:
from sklearn.model_selection import train_test_split

df_remaining["stratify_key"] = (
    df_remaining["movie_title"].astype(str) + "_" +
    df_remaining["author_rating"].astype(str)
)

train_df, test_df = train_test_split(
    df_remaining,
    test_size=0.2,
    random_state=42,
    stratify=df_remaining["stratify_key"]
)

train_df = train_df.drop(columns=["stratify_key"])
test_df = test_df.drop(columns=["stratify_key"])

In [23]:
print(len(train_df), len(test_df))

print(train_df["author_rating"].value_counts(normalize=True))
print(test_df["author_rating"].value_counts(normalize=True))

print(train_df["movie_title"].value_counts().head())
print(test_df["movie_title"].value_counts().head())

71448 17862
author_rating
10.0    0.238490
8.0     0.146239
9.0     0.128286
7.0     0.119202
6.0     0.090526
1.0     0.075031
5.0     0.068491
4.0     0.049661
3.0     0.045191
2.0     0.038881
Name: proportion, dtype: float64
author_rating
10.0    0.238399
8.0     0.146283
9.0     0.128227
7.0     0.119200
6.0     0.090334
1.0     0.075384
5.0     0.068599
4.0     0.049681
3.0     0.045253
2.0     0.038641
Name: proportion, dtype: float64
movie_title
Avengers: Endgame          7710
Captain Marvel             5916
Spider-Man: No Way Home    4957
Avengers: Infinity War     3734
Thor: Love and Thunder     3472
Name: count, dtype: int64
movie_title
Avengers: Endgame          1927
Captain Marvel             1475
Spider-Man: No Way Home    1237
Avengers: Infinity War      933
Thor: Love and Thunder      867
Name: count, dtype: int64


In [25]:
train_df.to_csv("data/train_df.csv", index=False)
test_df.to_csv("data/test_df.csv", index=False)

In [4]:
import polars as pl

df = pl.read_csv(r'..\data\mcu_imdb_reviews.csv')

In [5]:
# Assuming your text column is called "content"
text_col = "content"

# 1. Number of records
num_records = df.height

# 2. Split text into words
df_words = df.with_columns(
    pl.col(text_col)
    .str.to_lowercase()
    .str.replace_all(r"[^a-z0-9\s]", "")  # remove punctuation
    .str.split(" ")
    .alias("tokens")
)

# Explode into one word per row
tokens = df_words.select(pl.col("tokens").explode().alias("word"))

# Remove empty strings
tokens = tokens.filter(pl.col("word") != "")

# 2. Number of words (total tokens)
num_words = tokens.height

# 3. Number of types (unique words)
num_types = tokens.select(pl.col("word").n_unique()).item()

print(f"Records: {num_records}")
print(f"Words: {num_words}")
print(f"Unique words (types): {num_types}")

Records: 90310
Words: 16000913
Unique words (types): 125339
